# S1-2 · 注意力

`CausalSelfAttention`（`model.py` 第 29–76 行）是整个文件最值得反复看的地方。

**输入输出都是 `(b, t, n_embd)`**，它做的是：让序列里每个位置去「看」它前面所有位置，再把看到的信息混进自己的向量。

In [ ]:
import pathlib
import sys

cands = []
nb_file = globals().get("__vsc_ipynb_file__")
if nb_file:
    cands.append(pathlib.Path(nb_file).parent)
cands += [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]

ROOT = next((c for c in cands if (c / "vendor" / "nanoGPT" / "model.py").exists()), None)
assert ROOT is not None, "没找到 vendor/nanoGPT/model.py，请确认工作目录在项目内"

sys.path.insert(0, str(ROOT / "vendor" / "nanoGPT"))
import model as nanogpt

import torch
import torch.nn.functional as F

torch.manual_seed(0)
cfg = nanogpt.GPTConfig(block_size=16, vocab_size=11, n_layer=1, n_head=2, n_embd=8, dropout=0.0)
attn = nanogpt.CausalSelfAttention(cfg)

x = torch.randn(2, 6, 8)      # b=2, t=6, C=8
B, T, C = x.shape
nh = cfg.n_head               # 2 个头
hs = C // nh                  # 每头 4 维

print("输入 x：", tuple(x.shape))
print("输出   ：", tuple(attn(x).shape))

## 一、Q、K、V 一次算出来

```python
self.c_attn = nn.Linear(n_embd, 3 * n_embd)     # 第 35 行
...
q, k, v = self.c_attn(x).split(n_embd, dim=2)   # 第 56 行
```

**为什么一次算 3 倍宽？** Q、K、V 本质是三次矩阵乘，拼成一个 `(C, 3C)` 的矩阵一次算完，GPU 更划算，算完再切成三份。

三个东西的分工（查字典的比喻）：

| | 含义 | 类比 |
|---|---|---|
| **Q**（query） | 我现在要找什么 | 我要查的词 |
| **K**（key） | 我能提供什么 | 每个词条的标题 |
| **V**（value） | 我实际的内容 | 每个词条的正文 |

In [ ]:
qkv = attn.c_attn(x)
print("c_attn(x) ：", tuple(qkv.shape), " ← 最后一维是 3C = 24")

q, k, v = qkv.split(C, dim=2)
print("split 之后：q", tuple(q.shape), " k", tuple(k.shape), " v", tuple(v.shape))

## 二、分头

第 57–59 行：

```python
k = k.view(B, T, n_head, C // n_head).transpose(1, 2)   # (B, nh, T, hs)
```

从 `(2, 6, 8)` 变成 `(2, 2, 6, 4)`：切成 2 个头，每个头 4 维。

**为什么分头？** 让模型同时从多个角度观察同一段上下文，每个头可以学不同的关注模式（有的看相邻词，有的看语法结构）。

**注意顺序**：先 `view` 拆成 `(B, T, nh, hs)`，再 `transpose` 变成 `(B, nh, T, hs)`。

In [ ]:
q = q.view(B, T, nh, hs).transpose(1, 2)
k = k.view(B, T, nh, hs).transpose(1, 2)
v = v.view(B, T, nh, hs).transpose(1, 2)

print("分头之后：q", tuple(q.shape), " k", tuple(k.shape), " v", tuple(v.shape))
print("含义：2 个 batch × 2 个头 × 6 个位置 × 每头 4 维")

### 为什么不能直接 view 成 `(B, nh, T, hs)`

两种写法形状一样，**内容完全不同**，而且不报错。

`view` 只按内存顺序重新切片，它不知道你想表达什么语义。`(B,T,C)` 的内存顺序是「token 优先、特征其次」，`view(B, nh, T, hs)` 会按这个顺序硬切成「头优先」，把不同 token 的数据混在一起。

In [ ]:
k_flat = attn.c_attn(x).split(C, dim=2)[1].contiguous()

right = k_flat.view(B, T, nh, hs).transpose(1, 2)   # 正确：先拆 (nh, hs)，再换维度
wrong = k_flat.view(B, nh, T, hs)                   # 错误：直接按 (nh, T, hs) 切

print("正确：", tuple(right.shape))
print("错误：", tuple(wrong.shape), " ← 形状一模一样")
print("内容相同吗：", torch.equal(right, wrong))

In [ ]:
# split 出来的张量不一定「内存连续」，直接 view 可能报 stride 错误
_, k_split, _ = attn.c_attn(x).split(C, dim=2)
print("split 结果是否连续：", k_split.is_contiguous())
print("用 reshape 一定安全  ：", tuple(k_split.reshape(B, T, nh, hs).shape))

## 三、打分：Q 和 K 的点积

```python
att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
```

形状追踪：

```
q           (b, nh, t, hs)
k.T         (b, nh, hs, t)
q @ k.T     (b, nh, t, t)     ← 每个位置 × 每个位置的分数
```

`(b, nh, t, t)` 的第 `i` 行第 `j` 列 = 「位置 i 对位置 j 的关注程度」。

**为什么除以 `√hs`**：点积是 `hs` 项相加，方差随维度线性增长。不缩放的话分数会很大，softmax 进入饱和区（几乎全是 0 或 1），梯度几乎为 0，训练不动。除以 `√hs` 把方差拉回 1。

In [ ]:
import math

scores = q @ k.transpose(-2, -1)
print("q        ：", tuple(q.shape))
print("k.T      ：", tuple(k.transpose(-2, -1).shape))
print("scores   ：", tuple(scores.shape), " ← (b, nh, t, t)")

scaled = scores * (1.0 / math.sqrt(hs))
print("\n缩放前（batch 0, head 0）：")
print(scores[0, 0].round(decimals=2))
print("缩放后（除以 √4 = 2）：")
print(scaled[0, 0].round(decimals=2))

## 四、因果遮罩

第 68 行：

```python
att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
```

`bias` 是下三角矩阵（第 49–50 行）：

```
[[1, 0, 0, 0],
 [1, 1, 0, 0],
 [1, 1, 1, 0],
 [1, 1, 1, 1]]
```

**为什么必须下三角**：任务目标是「预测下一个词」。如果位置 0 能看到位置 3 的内容，等于提前看答案，训练就废了。

遮罩后加 softmax，`-inf` 自动变成概率 0。

In [ ]:
T4 = 4
mask_small = torch.tril(torch.ones(T4, T4))
print("遮罩（1 = 可以看，0 = 不许看）：")
print(mask_small)

fake = torch.randn(1, 1, T4, T4)
masked = fake.masked_fill(mask_small == 0, float("-inf"))
print("\n填 -inf 之后：")
print(masked[0, 0].round(decimals=2))

probs = torch.softmax(masked, dim=-1)
print("\nsoftmax 之后（-inf 变成 0）：")
print(probs[0, 0].round(decimals=2))
print("每行和：", probs[0, 0].sum(dim=-1).round(decimals=3).tolist())

In [ ]:
print("模型用的是 flash attention 吗：", attn.flash)
print("有显式 bias 缓冲吗：", hasattr(attn, "bias"))

## 五、softmax + 加权求和

```python
att = F.softmax(att, dim=-1)
att = self.attn_dropout(att)
y = att @ v
```

```
att    (b, nh, t, t)   ← 每行和为 1
v      (b, nh, t, hs)
att@v  (b, nh, t, hs)  ← 每个位置取到「按关注度加权平均后的内容」
```

**这就是注意力的本质**：每个位置按自己的需要，去前面所有位置取一个加权平均。

In [ ]:
mask_T = torch.tril(torch.ones(T, T))

att_full = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(hs))
att_full = att_full.masked_fill(mask_T == 0, float("-inf"))
att_full = torch.softmax(att_full, dim=-1)

print("att 形状：", tuple(att_full.shape))
print("每行和  ：", att_full[0, 0].sum(dim=-1).round(decimals=3).tolist())
print("\n第 0 个 batch 第 0 个头的注意力矩阵：")
print(att_full[0, 0].round(decimals=2))

y = att_full @ v
print("\ny = att @ v ：", tuple(y.shape))

## 六、合头 + 输出投影

```python
y = y.transpose(1, 2).contiguous().view(B, T, C)
y = self.resid_dropout(self.c_proj(y))
```

`(b, nh, t, hs)` → 换回 `(b, t, nh, hs)` → 拼成 `(b, t, C)`。

`contiguous()` 是必须的：`transpose` 之后内存不连续，`view` 会报错（B1 那个坑）。

In [ ]:
back = y.transpose(1, 2)
print("transpose 后：", tuple(back.shape), " 连续？", back.is_contiguous())

merged = back.contiguous().view(B, T, C)
print("view 之后   ：", tuple(merged.shape))

out = attn.c_proj(merged)
print("c_proj 之后 ：", tuple(out.shape), " ← 和输入形状一致")

In [ ]:
# 手工算的这一整套，和模块直接调用结果一致吗
print("与 attn(x) 一致：", torch.allclose(out, attn(x), atol=1e-6))

## 七、Flash Attention 分支

第 62–64 行：

```python
if self.flash:
    y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=..., is_causal=True)
```

`is_causal=True` 让 PyTorch 内部直接用高效的因果注意力 kernel，**不会真的把 `(t, t)` 矩阵建出来**，显存和速度都好很多。

所以第 67–71 行那段手写版本是兜底。但**原理要看手写那段**，理解之后再知道有加速实现即可。

In [ ]:
flash_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("flash 输出：", tuple(flash_out.shape))
print("与手写版本差多少：", (flash_out - y).abs().max().item())

## 八、留给你回答的三个问题

1. `c_attn` 为什么做成 `3 * n_embd` 宽的一个 Linear，而不是三个独立的 Linear？
2. 为什么必须 `view(B, T, nh, hs).transpose(1, 2)`，而不能直接 `view(B, nh, T, hs)`？
3. 分数为什么要除以 `√hs`？不除会发生什么？

答完这三个，注意力就算通了。下一步是跑通 `shakespeare_char`。